## Preprocessing

In [1]:
# Importing the dataset
import pandas as pd

df = pd.read_csv("../data/working.csv")

In [2]:
# selcting only needed features from the dataset
selected_features = [
    "URLLength",
    "IsDomainIP",
    "TLD",
    "NoOfDegitsInURL",
    "NoOfOtherSpecialCharsInURL",
    "IsHTTPS",
    "DomainLength",
    "URLSimilarityIndex",
    "NoOfSubDomain",
    "HasObfuscation",
    "label"
]

df = df[selected_features]

In [3]:
# seperate the solution (if phishing or legitimate url) from the rest of the data
X = df.drop("label", axis=1)
y = df["label"]

In [4]:
print(df.columns)
print(X.columns)
print(y.name)

Index(['URLLength', 'IsDomainIP', 'TLD', 'NoOfDegitsInURL',
       'NoOfOtherSpecialCharsInURL', 'IsHTTPS', 'DomainLength',
       'URLSimilarityIndex', 'NoOfSubDomain', 'HasObfuscation', 'label'],
      dtype='object')
Index(['URLLength', 'IsDomainIP', 'TLD', 'NoOfDegitsInURL',
       'NoOfOtherSpecialCharsInURL', 'IsHTTPS', 'DomainLength',
       'URLSimilarityIndex', 'NoOfSubDomain', 'HasObfuscation'],
      dtype='object')
label


In [5]:
# Checking how many different tld's there are
df["TLD"].nunique()

695

In [6]:
# looking at the top 10 most prevalant
top_tlds = X["TLD"].value_counts().nlargest(10).index
print(top_tlds.values)
X["TLD"].value_counts()

['com' 'org' 'net' 'app' 'uk' 'co' 'io' 'de' 'ru' 'au']


TLD
com           112554
org            18793
net             7097
app             6508
uk              6395
               ...  
15                 1
nr                 1
parts              1
kitchen            1
associates         1
Name: count, Length: 695, dtype: int64

In [7]:
# replacing every other tld with other to limit the amount of different values to 11:
X["TLD"] = X["TLD"].apply(lambda x: x if x in top_tlds else "other")
X["TLD"].nunique()

11

In [8]:
# creating new features to encode the TLD feature into 1 and 0 values for each common top level domain:
X = pd.get_dummies(X, columns=["TLD"])
print(X.columns)
print(len(X.columns))

Index(['URLLength', 'IsDomainIP', 'NoOfDegitsInURL',
       'NoOfOtherSpecialCharsInURL', 'IsHTTPS', 'DomainLength',
       'URLSimilarityIndex', 'NoOfSubDomain', 'HasObfuscation', 'TLD_app',
       'TLD_au', 'TLD_co', 'TLD_com', 'TLD_de', 'TLD_io', 'TLD_net', 'TLD_org',
       'TLD_other', 'TLD_ru', 'TLD_uk'],
      dtype='object')
20


## Data split

In [9]:
# splitting data into training and testing sets. X is data and y is for evaluation
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(188636, 20) (47159, 20)


In [11]:
# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
# training the first model
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)

In [15]:
# evaluation

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.9996395173773829
[[20172    17]
 [    0 26970]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



In [16]:
import joblib

joblib.dump(model_lr, "../models/linear_model.pkl")

['../models/linear_model.pkl']

In [18]:
# cross validation to ensure the result is stable
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model_lr, X, y, cv=5)

print(scores.mean())

0.9997837104264298


## conclusion
while this model seems to be very accurate (almost too accurate - only 17 mistakes out of 47,159), after trying to use it, i figured that it is not practical. Especially the URLSimilarityIndex feature is not something users can simply derive from the url they want to check. So I'll make another model without this feature.